# Lesson 68 · How CTC alignment works — how do 100 frames come out as 5 letters? The blank symbol and label collapse

**Goal**: think through CTC's blank, collapse and multiple paths first, then answer "how do 100 frames come out as 5 letters?"

> **Opening with the pain point**: how do you align when you don't know where the boundaries are? Look at the picture first, then the formula; build an intuition for path compression with greedy decoding first.

🔗 **Aurora connection**: `aurora.speech` will call `torch.nn.CTCLoss`; understanding CTC is what lets you see why Whisper chose a different training objective. In this section we hand-write greedy decoding to build intuition for CTC path compression.

← **Previous lesson** · [L67 · Edit distance from scratch](L67_edit_distance.ipynb)

> Last lesson covered **edit distance from scratch**: Levenshtein dynamic programming, the maths of WER.
> This lesson explores **how CTC alignment works**.

## This lesson's story: the model makes 100 predictions a second, but the sentence has only 5 letters — is that allowed?

A speech-recognition model outputs one token prediction per frame — about 100 frames for 1 second of audio, while the target text might be only 5 characters. How does the model squeeze those frames down to the final 5 characters?

**The answer of CTC (connectionist temporal classification)**: introduce a blank symbol `∅`.

Imagine typing on your phone with a laggy keyboard: you press `a-a-a-∅-b-b-∅-c`, and the phone outputs `abc` — that's CTC.

**CTC path → target sequence, in two compression steps**:
```
path π  = [a, a, ∅, b, b, ∅, ∅, c]   (the model's frame-level output, with repeats and blanks)
step 1  → [a, ∅, b, ∅, ∅, c]           remove adjacent repeats (a-a→a, b-b→b)
step 2  → [a, b, c]                     remove blanks
```

**Why CTC works**: one target sequence `"abc"` corresponds to an **exponential** number of paths (every combination of blank positions is legal), and the model sums the probabilities of all those paths to get the total probability of the output sequence. Training maximizes this total probability, with no need to label in advance "which frame goes with which character".

**Greedy decoding**: take the most probable token in each frame, then compress. Fast, but not guaranteed optimal:

```
logits (T, V) → argmax per frame → path π → collapse() → target sequence
```

This section's task: implement `ctc_greedy_decode(logits, blank=0)` and check that it gives the right output on known logits.

## 🤔 Why did engineers invent this?

- **What happens without it?** You'd first have to label by hand "frame 37 is the letter h, frame 38 is still h, frame 39 is e……" — all 100 frames of every second aligned by hand, at a cost far too high to train at scale.
- **What real problem does it solve?** Speech has **far more frames than characters**, and there are no frame-by-frame labels. CTC lets you give only "this audio = this sentence" (a sequence-level label), and the model works out for itself which frames each character lines up with.
- **Where will it come up again?** L69 (the forward algorithm sums these paths into a loss), L70+ (Whisper switches to cross-entropy, but the alignment problem of "many frames, few characters" is the same one).

### The smallest example: many-to-one (many frames → few letters)

The model emits one symbol per frame (`_` = blank). 5 frames of output → collapse (**first remove adjacent repeats, then remove blanks**) → 2 letters:

```
A A _ B B   →  remove repeats→ A _ B  →  remove blanks→  A B   ✅
```

**Key intuition**: many different frame sequences all collapse to the same `"AB"` — that's **many-to-one**:

```
A A _ B B   →  AB
_ A _ B _   →  AB
A _ _ _ B   →  AB
A A A B B   →  AB   (AA→A, BB→B)
```

Training doesn't pick one of these; it **adds up** the probabilities of **all** the paths that collapse to `"AB"` (which is exactly what L69 does).

## ⚠️ Common misconceptions

> Don't read the blank `_` as "an empty character / a space / silence". It's a placeholder for **"don't emit a letter just yet"** — purely an **alignment tool**. It has two jobs: (1) giving the extra frames somewhere to go when there are fewer characters than frames; (2) separating two identical characters — `A _ A` collapses to `AA` (two A's), while `A A` collapses to a single `A`.


## Sequence-level vs frame-level labels: why is no alignment needed?

**The traditional approach (needs frame-level labels, expensive)**:

```
audio waveform (1 s = 100 frames):    [──── ──── ──── ...]
hand labels (frame by frame):         h    o    l    l    o    ∅    ∅    ...
every frame must be labeled with its letter, a huge amount of work (all 100 frames)
```

**CTC's clever trick (only sequence-level labels needed, cheap)**:

```
audio waveform (1 s = 100 frames):    [──── ──── ──── ...]
sequence-level label (whole):                 "hello"
the model works out the alignment itself!
```

**Why does this work?** — because CTC encodes **every possible** alignment through "the total probability summed over paths":

```
target = "ab"

model output: probability distributions over 3 frames
  frame 1: a=80%, ∅=20%
  frame 2: b=70%, ∅=30%
  frame 3: ∅=60%, b=40%

legal paths (all collapse to "ab"):
  [a, b, ∅]  probability = 0.8 × 0.7 × 0.6 = 0.336
  [a, ∅, b]  probability = 0.8 × 0.3 × 0.4 = 0.096
  [a, b, b]  probability = 0.8 × 0.7 × 0.4 = 0.224

P(y="ab" | audio) = Σ probabilities of all legal paths
                  = 0.336 + 0.096 + 0.224
                  = 0.656 (these three are the only legal paths, since only frame 1 can produce a)
```

**How the model learns**:
- During training, maximize P(y="ab" | x=audio)
- That amounts to having the model spread probability over every alignment that yields "ab"
- The model learns automatically that the real alignment of some "ab" could be [a,b,∅] or [a,∅,b] — both are reasonable
- No need for hard constraints specified by hand, like "a is in frame 1" and "b is in frame 2"

**Side-by-side**:
| Method | Labeling cost | Alignment flexibility | Training data needed |
|---|---|---|---|
| Frame-level labels | high (hand-labeled per frame) | fixed (just one alignment) | less (strong constraints) |
| CTC (sequence-level) | low (label the sequence only) | flexible (every legal alignment) | more (larger space to learn) |

That's why CTC was revolutionary for speech recognition — it slashed the cost of labeling.

In [ ]:
import numpy as np

## 1. CTC paths and path probabilities

Let the input sequence have length `T` and the vocabulary have size `V` (including blank=0).
At every frame `t` the model outputs a probability distribution `p(token | t)`, of shape `(T, V)`.

A **CTC path** `π` is a token sequence of length `T` (one symbol per frame, blanks included).
The path's probability is the product of the per-frame probabilities:

```
P(π) = ∏_{t=1}^{T}  p(π_t | t)
```

The **CTC loss** maximizes the sum of the probabilities of all paths that "collapse to the target `y`":

```
P(y | x) = Σ_{π : collapse(π)=y}  P(π)
```

The number of paths is exponential, so in practice it's computed exactly with the **forward–backward algorithm**, a dynamic program (DP).

In [ ]:
# Demo: 3 frames, vocabulary = {blank=0, a=1, b=2}
# Build a simple log-probability matrix (specified by hand)
T, V = 3, 3
log_probs = np.array([
    [np.log(0.1), np.log(0.8), np.log(0.1)],  # frame 0: a is largest
    [np.log(0.7), np.log(0.2), np.log(0.1)],  # frame 1: blank is largest
    [np.log(0.1), np.log(0.1), np.log(0.8)],  # frame 2: b is largest
])
# List two paths, (a, blank, b) and (a, a, b); both collapse to [a, b]
path1 = [1, 0, 2]  # a ∅ b  -> collapse -> a b
path2 = [1, 1, 2]  # a a b  -> collapse -> a b
p_path1 = np.exp(log_probs[0,1] + log_probs[1,0] + log_probs[2,2])
p_path2 = np.exp(log_probs[0,1] + log_probs[1,1] + log_probs[2,2])
print(f'P(a ∅ b) = {p_path1:.4f}')
print(f'P(a a b) = {p_path2:.4f}')
print(f'P(y=[a,b] | x) ≈ {p_path1 + p_path2:.4f}  (only two paths enumerated)')


## 2. The collapse rule

Given a path `π` (blanks included), it collapses to the target sequence in two steps:

```
step 1: remove adjacent repeats  [a, a, ∅, b, b, ∅, c] → [a, ∅, b, ∅, c]
step 2: remove blanks            [a, ∅, b, ∅, c]       → [a, b, c]
```

Note that one job of the blank is to **separate identical characters**:
the path `[a, a, b]` collapses to `[a, b]`;
the path `[a, ∅, a, b]` collapses to `[a, a, b]` (with a blank between the two a's, they don't merge).

In [ ]:
# collapse written as a plain for loop (if list comprehensions are hard to read, this is the one for you)
def collapse_with_loop(path, blank=0):
    """CTC path collapse: remove adjacent repeats first, then remove blanks (with for loops, easy to follow step by step).

    Args:
        path: list[int], the frame-level symbol sequence (with blanks)
        blank: token id of the blank symbol, default 0
    Returns:
        list[int], the collapsed sequence (no blanks, adjacent repeats merged)
    """
    # ===== Step 1: remove adjacent repeats =====
    deduped = []
    for i, token in enumerate(path):
        # keep it only when i=0 (the first one) or the current token ≠ the previous token
        if i == 0 or token != path[i - 1]:
            deduped.append(token)

    # ===== Step 2: remove blanks =====
    final = []
    for token in deduped:
        if token != blank:
            final.append(token)

    return final

# Check that this version and the list-comprehension version later give the same result
test_path = [1, 1, 0, 2, 2, 0, 3]
print(f"For-loop version: collapse_with_loop({test_path}) = {collapse_with_loop(test_path)}")


## Why must you **remove adjacent repeats first, then blanks**? Can the order be swapped?

This is a key design choice. Let's use a counterexample to see why the order can't be reversed:

```
path π = [a, ∅, a, b]   # a blank sits between the two a's

[Correct order] remove repeats first → then blanks:
  [a, ∅, a, b]  →  remove adjacent repeats  →  [a, ∅, a, b]  (∅ in the middle, the a's aren't adjacent, kept)
              →  remove blanks  →  [a, a, b]  ✓ both a's survive

[Wrong order] remove blanks first → then repeats:
  [a, ∅, a, b]  →  remove blanks  →  [a, a, b]
              →  remove adjacent repeats  →  [a, b]  ✗ the two a's wrongly merge into one!
```

**The core rule**: one job of `∅` is to **separate two identical characters** so they don't get merged by mistake.
- `[a, a, b]` → `[a, b]` (aa adjacent, merged into one a)
- `[a, ∅, a, b]` → remove repeats first (∅ kept, the a's not adjacent) → `[a, ∅, a, b]`; then remove blanks → `[a, a, b]` (two a's)

So **the definition of collapse must be "remove repeats first, then blanks"**. The order can't change; changing it would give different collapse results, and the training objective would be completely different.

### Path probability: chaining per-frame predictions together with multiplication

At each frame `t` the model outputs one number per token, called the **logits** or **scores**.
After a softmax they become probabilities. For example, if the model's output at frame 3 is `[0.6, 0.3, 0.1]`, it means "at frame 3, token-0 has probability 60%, token-1 30%, token-2 10%".

The **conditional probability notation** `p(π_t | t)` means something very plain:
- The vertical bar `|` means "given that……"
- `p(π_t | t)` = "the probability that, at frame `t`, the predicted symbol is π_t"
- A school-maths analogy: `p(rain | cloudy)` = "the probability of rain when it's cloudy"

A path `π = [token₁, token₂, ..., tokenₜ]` is stitched together from **many frames**, and CTC treats the per-frame predictions as **independent of each other** given the audio (each frame's prediction is made separately), so the total probability is the **product** of the per-frame probabilities:

```
P(π) = p(π₁|frame 1) × p(π₂|frame 2) × ... × p(πₜ|frame T)
     = ∏_{t=1}^{T}  p(π_t | t)
```

Notation:
- `∏` means repeated multiplication `×` (a product)
- `Σ` means summation `+`
- `∏_{t=1}^{T}` means multiplying from frame t=1 through t=T

### Why a **sum** and not a **max**?

Intuitively, one target sequence `y` (say "ab") corresponds to many different frame-level paths.
The greediest idea is "pick only the most probable path", i.e.:
```
P_greedy(y|x) = max_{π: collapse(π)=y}  P(π)
```

But CTC's design is to **add up the probabilities of every path that collapses to y**:
```
P_CTC(y|x) = Σ_{π: collapse(π)=y}  P(π)
```

**Why sum instead of max?** — because that's **the correct use of probability**:
- Think of rolling a die. The probability of "4 or more" = P(4) + P(5) + P(6) = 1/6 + 1/6 + 1/6 = 1/2
- Not max(P(4), P(5), P(6)) = 1/6
- If a target sequence can come from several paths (rolling a 4, 5 or 6 all count as "≥4"), you should add those probabilities up

**A picture of the comparison**:
```
From the path-probability view:
  path 1 [a,∅,b]  probability 0.3
  path 2 [a,a,b]  probability 0.2    } all collapse to "ab"
  path 3 [∅,a,b]  probability 0.1    }

greedy: P(y="ab") = max(0.3, 0.2, 0.1) = 0.3
CTC:    P(y="ab") = 0.3 + 0.2 + 0.1 = 0.6  ✓ this is the true probability that "y="ab" happens"
```

Maximizing P_CTC during training means: **encourage the model to give high probability to every legal path**, rather than betting on a single one. That's more robust.

### Exponentially many paths: why does one target sequence have so many paths?

**A concrete example**: target = `"ab"`, T=4 frames, vocabulary V={0:∅, 1:a, 2:b}

Which frame-level paths are legal? Anything that "collapses to ab" will do:
```
[a, ∅, b, ∅]  → a b    ✓
[a, a, b, ∅]  → a b    ✓  (aa → a, giving ab)
[a, ∅, b, b]  → a b    ✓  (bb → b)
[a, ∅, ∅, b]  → a b    ✓
[a, a, ∅, b]  → a b    ✓
```

Why so many? Because:
1. **Blanks can be inserted anywhere**: a can be followed by any number of blanks or more a's (removed when deduplicating)
2. **Repeated characters can sit together**: `aa` → `a` (still a after deduplication)

**The path-count formula**: with T input frames and a vocabulary of V symbols (blank included), every path is a sequence of T tokens.
Each frame chooses independently, so there are **V^T** possible frame-level paths in total. Most of them collapse to different target sequences, but one target sequence can have hundreds or even thousands of paths behind it.

**A numerical example**:
- T=5 frames, V=4 (blank + 3 tokens) → 4^5 = 1024 paths in total
- Of these, maybe 50 collapse to "ab", 100 to "abc", and so on
- With greedy (looking only at the single best path) you'd miss the 49 sub-optimal paths that also support "ab"

That's why **CTC can't just take the max**; it has to sum over every legal path — each one contributes.

In [ ]:
# Actually counting: how many legal paths does one target sequence have?
from itertools import product

def count_ctc_paths(target_labels, T, blank=0):
    """Enumerate every path of length T and count how many collapse to target_labels.

    Args:
        target_labels: list[int], the target label sequence (no blanks)
        T: path length (number of frames)
        blank: token id of the blank symbol

    Returns:
        count: number of legal paths
        example_paths: the first 10 legal paths, as examples
    """
    V = len(set(target_labels) | {blank}) + 1  # a guess at the vocabulary size (widened below)
    valid_paths = []

    # Enumerate every possible path
    for path in product(range(V + 5), repeat=T):  # brute force, but it works
        if collapse_with_loop(list(path), blank=blank) == target_labels:
            valid_paths.append(path)

    return len(valid_paths), valid_paths[:10]

# Example
target = [1, 2]  # a, b
T_test = 4
count, examples = count_ctc_paths(target, T_test, blank=0)
print(f"target = {target}, frames T = {T_test}")
print(f"total legal paths = {count}")
print(f"the first 5 example paths:")
for ex in examples[:5]:
    print(f"  {list(ex)} → {collapse_with_loop(list(ex))}")

print(f"\nComparison:")
print(f"upper bound on total paths (V^T, vocabulary V=3) = 3^{T_test} = {3**T_test}")
print(f"actual legal paths = {count}")
print(f"ratio = {count / (3**T_test):.1%}")


In [ ]:
def collapse(path, blank=0):
    """CTC path collapse: remove adjacent repeats first, then remove blanks.

    [Compact version] using list comprehensions
    - If list comprehensions are hard to read, see collapse_with_loop in an earlier cell; the logic is exactly the same
    - The two versions are equivalent; this is just a more compact way to write it
    """
    # Step 1: remove adjacent repeats
    # Meaning: [p for i, p in enumerate(path) if i == 0 or p != path[i-1]]
    #      = keep element 0, or any element that differs from the one before it
    deduped = [p for i, p in enumerate(path) if i == 0 or p != path[i-1]]

    # Step 2: remove blanks
    # Meaning: [p for p in deduped if p != blank]
    #      = keep every element that isn't blank
    return [p for p in deduped if p != blank]

# Check on examples
examples = [
    ([1, 1, 0, 2, 2, 0, 3], 'a a ∅ b b ∅ c'),
    ([1, 0, 1, 2],           'a ∅ a b'),
    ([1, 1, 2],              'a a b'),
    ([0, 0, 1, 0],           '∅ ∅ a ∅'),
]
for path, desc in examples:
    result = collapse(path)
    print(f'  [{desc}]  →  {result}')

## 3. Greedy decoding

Better CTC decoding usually uses beam search (enumerating every path is too slow).
**Greedy decoding** is the fastest approximation: take the argmax at every frame to get one path, then collapse it.

```
π_greedy = [argmax p(token | t)  for t in 1..T]
y_greedy = collapse(π_greedy)
```

Greedy decoding isn't optimal — it considers only the single most probable path, while the right answer may be held up jointly by several sub-optimal paths.
But in practice, in CTC systems such as DeepSpeech and wav2vec 2.0, it already works well as a fast baseline.

In [ ]:
# Demo: greedy decoding
np.random.seed(42)
T_demo, V_demo = 7, 5  # 7 frames, vocabulary size 5 (0=blank, 1=a, 2=b, 3=c, 4=d)
logits_demo = np.random.randn(T_demo, V_demo)
# Push some frames up by hand so the result is easier to read
logits_demo[0, 1] = 5   # frame 0: a
logits_demo[1, 0] = 5   # frame 1: blank
logits_demo[2, 0] = 5   # frame 2: blank
logits_demo[3, 2] = 5   # frame 3: b
logits_demo[4, 2] = 5   # frame 4: b
logits_demo[5, 0] = 5   # frame 5: blank
logits_demo[6, 3] = 5   # frame 6: c

preds = np.argmax(logits_demo, axis=1)
vocab = {0:'∅', 1:'a', 2:'b', 3:'c', 4:'d'}
print('frame-level argmax    :', [vocab[p] for p in preds])
print('adjacent repeats gone :', [vocab[p] for i,p in enumerate(preds) if i==0 or p!=preds[i-1]])
decoded = collapse(preds.tolist())
print('final decoded result  :', [vocab[p] for p in decoded], '→ token ids:', decoded)


## 4. ✏️ Implement `ctc_greedy_decode(logits, blank=0)`

**Signature**:
```python
def ctc_greedy_decode(logits: np.ndarray, blank: int = 0) -> list:
    # logits: (T, V) numpy array (raw scores before softmax)
    # returns: list[int], the collapsed token-id sequence (no blanks)
```

**A 3-step implementation**:

| Step | Operation | Tool |
|---|---|---|
| 1 | `path = logits.argmax(axis=1)` | take the top token in each frame |
| 2 | remove adjacent repeats: `[a,a,b,b] → [a,b]` | a loop or `itertools.groupby` |
| 3 | remove blanks: filter out the blank token | list comprehension |

**Acceptance criteria**:
- all-blank logits → `[]` (an empty list)
- `[a, a, ∅, b, ∅]` → `[a, b]`
- `[a, ∅, a]` → `[a, a]` (a blank between the two a's, so they don't merge)

### Note: logits vs log_probs

In the earlier demo we built `log_probs = np.log(probabilities)` (the log of probabilities) by hand.
But a real neural network outputs **logits** (raw scores before softmax), not probabilities.

```
logits (raw network output):      [5.2, -1.3, 0.8]   (any real numbers, positive or negative)
                                  ↓ softmax
probabilities (a distribution):   [0.98, 0.01, 0.01]  (sum = 1, all between 0 and 1)
                                  ↓ log
log probabilities:                [-0.02, -4.6, -4.8] (negative or 0)
```

**ctc_greedy_decode takes logits (raw scores), not log probabilities**.
Why does argmax work for both? Because `softmax` and `log` are both **monotonically increasing**:

```
argmax(logits) = argmax(softmax(logits)) = argmax(log(softmax(logits)))

so the position of the largest logit = the position of the largest probability = the position of the largest log probability
```

You don't need any explicit conversion; just take the argmax of the logits.

### Coming up: L69 uses the log domain and logaddexp

When you compute **a sum of probabilities over many paths**, you run into numerical trouble:

```python
# Multiplying many small probabilities directly underflows
p1, p2 = 1e-100, 1e-100
total = p1 + p2  # in log space this needs a "log addition"

# L69's fix: work in log space
log_p1, log_p2 = np.log(p1), np.log(p2)
log_total = np.logaddexp(log_p1, log_p2)  # the addition formula in the log domain
# np.logaddexp(a, b) = log(exp(a) + exp(b)), computed without underflow or overflow
```

For now you only need to know that greedy decoding avoids this complication (it doesn't sum anything); L69 covers it in detail.

### When does greedy decoding go wrong?

**The definition of greedy**: at every frame, independently take the most probable token, then collapse.

**When it fails**: when the best target sequence is held up jointly by several sub-optimal paths, while greedy looks at only one path, the probability of greedy's answer can be far below the best value.

**A concrete numerical example**:

```
target = [a], 2 frames
vocabulary = {0: ∅, 1: a}

probabilities (after softmax):
frame 0: token-0(∅) = 60%   token-1(a) = 40%
frame 1: token-0(∅) = 60%   token-1(a) = 40%

[Greedy decoding]:
frame 0: argmax = 0(∅)
frame 1: argmax = 0(∅)
path = [∅, ∅] → collapse → []   ✗ wrong! we expected [a]
greedy path probability = 0.6 × 0.6 = 0.36

[Paths that collapse to [a]]:
path 1 = [a, a] → collapse → [a]  ✓  P = 0.4 × 0.4 = 0.16
path 2 = [a, ∅] → collapse → [a]  ✓  P = 0.4 × 0.6 = 0.24
path 3 = [∅, a] → collapse → [a]  ✓  P = 0.6 × 0.4 = 0.24

sum of the three correct paths = 0.16 + 0.24 + 0.24 = 0.64

So although the greedy path ([∅,∅]) has the highest individual probability (0.36),
it is the only path that collapses to []; the vote for [a] is split across three paths, which together reach 0.64 > 0.36.
Greedy wrongly outputs [] instead of [a].
```

**Conclusion**: greedy decoding is fast but not guaranteed optimal. Beam search (exploring several high-probability paths and merging them) or the full forward algorithm (L69) is needed for an exact answer.
In practice, CTC models such as DeepSpeech and wav2vec 2.0 do fine with greedy decoding once trained (because training pushes up the probability of the correct answer's paths).

In [ ]:
def ctc_greedy_decode(logits: np.ndarray, blank: int = 0) -> list:
    """CTC greedy decoding: argmax per frame, remove adjacent repeats, remove blanks.

    Args:
        logits: shape (T, V) numpy array (raw scores before softmax)
        blank:  token id of the blank symbol, default 0
    Returns:
        list of decoded token ids
    """
    # ✏️ TODO: step 1 — take the argmax in each frame
    preds = None

    # ✏️ TODO: step 2 — remove adjacent repeats (keep the first, then keep only tokens that differ from the previous one)
    deduped = None

    # ✏️ TODO: step 3 — remove blanks and return list[int]
    raise NotImplementedError("Finish the three TODO steps above first, then run the check cell; if you get stuck, see solutions/L68_ctc_alignment_solutions.md")

In [ ]:
# --- Check ctc_greedy_decode ---

V_check = 5
logits_check = np.full((7, V_check), -10.0)
forced_ids = [1, 0, 0, 2, 2, 0, 3]
for t, tok in enumerate(forced_ids):
    logits_check[t, tok] = 10.0

try:
    result = ctc_greedy_decode(logits_check, blank=0)
    assert result == [1, 2, 3], f'expected [1,2,3], got {result}'
    print('✅ ctc_greedy_decode([a ∅ ∅ b b ∅ c]) =', result, '= [a, b, c]')

    # Edge case: all-blank input → empty list
    logits_blank = np.zeros((5, V_check))
    logits_blank[:, 0] = 10.0
    assert ctc_greedy_decode(logits_blank) == [], 'all-blank input should return an empty list'
    print('✅ all-blank input → []')

    # Edge case: a single frame, a single character
    logits_single = np.array([[-1.0, 5.0, -1.0]])
    assert ctc_greedy_decode(logits_single) == [1]
    print('✅ single frame with argmax=1 → [1]')

except (NotImplementedError, TypeError):
    print('⬜ Implement the three TODO steps of ctc_greedy_decode first, then run this check cell')


## 5. Parameter experiment: sequence compression rate under random logits

**Setup**:
- `T = 50` (input frames), `vocab_size = 30` (including blank=0)
- Run 1000 trials, each time sampling random `logits ~ N(0,1)`
- Record the mean and spread of `len(decoded) / T`

**What to expect**:
- With every frame random, the chance of picking blank is about `1/30 ≈ 3.3%`, and the chance of an adjacent repeat is about `1/30`
- After removing repeats and blanks, the average output length is `~93–94%` of the input
  - Derivation: the chance a frame picks the same token as the previous one is about 1/V, so the survival rate after deduplication is about (V-1)/V;
    then remove blanks (chosen with probability 1/V); treating the two steps as independent,
    the expected compression rate ≈ ((V-1)/V)² = (29/30)² ≈ 0.934
- A trained speech model predicts blank a great deal, so its compression rate is **far below** this random baseline (more blanks mean shorter output)

## Deriving the compression-rate formula: where does ((V-1)/V)² come from?

**The question**: with random logits (each frame picks a random token), how short is the output after "remove repeats + remove blanks"?

**Assumptions**:
- The vocabulary has V symbols (blank included); with random logits every token is chosen with probability 1/V
- Removing repeats and removing blanks are **independent** (a key assumption, discussed below)

**Derivation**:

### Step 1: survival rate when removing adjacent repeats

What's the probability that the current frame's token differs from the previous frame's?

```
the previous frame picked some token A (probability 1/V)
probability the current frame picks a different token = 1 - P(current frame = A)
                    = 1 - 1/V
                    = (V-1)/V
```

Intuition: of the V choices, V-1 are "different from the previous frame".

**Example**: V=30 (blank + 29 characters), P(different from the previous frame) = 29/30 ≈ 0.967

### Step 2: survival rate when removing blanks

What's the probability that the current frame is **not blank**?

```
P(token ≠ blank) = 1 - P(token = blank)
                 = 1 - 1/V
                 = (V-1)/V
```

Same logic: of the V choices, V-1 are not blank.

### Step 3: total survival rate, treating the two steps as **independent**

If the event A, "the frame survives deduplication", and the event B, "the frame itself is not blank", are **independent**, then:

```
P(a frame survives both steps) = P(A) × P(B)
                               = (V-1)/V × (V-1)/V
                               = ((V-1)/V)²
```

Expected output length = input length T × ((V-1)/V)²

**Checking the numbers**: V=30
```
expected compression rate = (29/30)² = 0.9667² ≈ 0.9344 ≈ 93.4%

that is: for 50 input frames, expect about 50 × 0.934 ≈ 46.7 output tokens
```

### ⚠️ A word on the independence assumption

Strictly speaking, "the two steps are independent" isn't quite right. In the random model:
- **Step 1** (remove repeats): the probability that "current frame ≠ previous frame" is (V-1)/V
- **Step 2** (remove blanks): the probability that "current frame ≠ blank" is (V-1)/V

Strictly, these two events are linked behind the scenes — for example, "the current frame is blank" implies "the current frame ≠ (a previous frame that isn't blank)".
But when V is large (like V=30), the link is weak, and the independence approximation is very accurate (relative error < 2%).

For a rigorous proof, use the joint probability:
```
P(survives both steps) = P(current ≠ previous AND current ≠ blank)

With random logits, the current frame has V equally likely choices. The previous frame is some token; blank is another token (or the same one):
- if the previous frame ≠ blank: V-2 choices (excluding the previous token and blank) → P = (V-2)/V
- if the previous frame = blank: V-1 choices (excluding blank / the previous token) → P = (V-1)/V

the probability that the previous frame is blank = 1/V; that it isn't = (V-1)/V
joint: P = (V-1)/V × (V-2)/V + 1/V × (V-1)/V = (V-1)/V × ((V-2)/V + 1/V) = (V-1)/V × (V-1)/V = ((V-1)/V)²

Huh, it comes out the same! So for random logits the independence approximation is actually **exact**.
```

**Conclusion**: the formula ((V-1)/V)² is exact for random logits; after real training, blank is predicted a great deal and the compression rate is lower.

In [ ]:
try:
    np.random.seed(0)
    T_exp, V_exp, N = 50, 30, 1000

    ratios = []
    for _ in range(N):
        logits_exp = np.random.randn(T_exp, V_exp)
        decoded_exp = ctc_greedy_decode(logits_exp, blank=0)
        ratios.append(len(decoded_exp) / T_exp)

    ratios = np.array(ratios)
    print(f'input length T = {T_exp}, vocabulary size V = {V_exp} (including blank)')
    print(f'1000 random trials:')
    print(f'  mean output length          : {ratios.mean()*T_exp:.1f} tokens')
    print(f'  mean compression rate len/T : {ratios.mean():.3f}')
    print(f'  min/max compression rate    : {ratios.min():.3f} / {ratios.max():.3f}')
    print()
    expected = ((V_exp - 1) / V_exp) ** 2
    print(f'expected compression-rate formula ((V-1)/V)² = ({V_exp-1}/{V_exp})² ≈ {expected:.3f}')
    print('(two-step independence approximation: remove adjacent repeats, then blanks; with random logits both steps are of the same size)')

except (NotImplementedError, TypeError):
    print('⬜ Implement ctc_greedy_decode first, then run the parameter experiment')


## 6. Aside: CTC's monotonic path constraint

**The constraint**: a CTC path π must satisfy: frames are processed in order from left to right, and the labels (non-blank symbols) also appear in strictly forward order — never going back.

```
A valid monotonic path:
  frame 0 → a (label 1)
  frame 1 → a (still label 1; blanks are skipped and don't count)
  frame 2 → blank
  frame 3 → b (label 2, moving forward)
  frame 4 → b (still label 2)
  ✓ the label sequence is a → a → b, non-decreasing

An invalid non-monotonic path (not allowed by the CTC algorithm):
  frame 0 → a
  frame 1 → b
  frame 2 → a  ✗ jumped back to an earlier label! CTC forbids it
```

**Why is this constraint needed?**

Physically, speech is a time series, and characters are pronounced from left to right in the order they're written.
You can't say the "o" first and then go back to say the "h" (the h in hello has to come first).
CTC turns this fact into an algorithmic constraint: the label axis never goes backwards, so the alignment stays consistent with the timing of the speech.

**Is monotonicity automatically satisfied by greedy decoding?**

Yes! The reason is simple:
- Greedy decoding processes frames one by one from 0 to T-1
- collapse only "removes repeats" and "removes blanks"; it **never changes the order**
- So the labels in the path necessarily stay in left-to-right order

```
For example:
frame sequence  [a, blank, b, b, blank, c]
                 ↓ collapse (order unchanged)
output          [a, b, c]   ✓ monotonic

Something like [c, a, b] can't happen, because collapse is an order-preserving filter
```

**Why does the L69 forward algorithm need to model monotonicity explicitly?**

Exact CTC has to **enumerate and sum over every legal path**. Not all V^T paths are legal — you have to pick out the ones that satisfy the monotonic constraint.
The forward algorithm uses dynamic programming, explicitly tracking "we're at frame t and have expanded up to position s in the label sequence", which guarantees monotonicity automatically.
Without this constraint the path space would be far bigger and the computation would explode.

**Summary**: the monotonic path constraint is a **basic assumption** of CTC, not an option. Greedy decoding satisfies it automatically; the exact algorithm uses it explicitly to shrink the search space.

## Appendix C · From greedy decoding to the forward algorithm: an L69 preview (8 minutes)

1. **What L68 did**: find **one** best path (the greedy approximation).
2. **What L69 will do**: **sum** the probabilities of **all** legal paths (the exact training loss).
3. **Why DP is needed**: there are O(V^T) paths; DP brings it down to O(T·S), where `S = 2L + 1`.
4. **How it relates to L67**: the edit-distance DP also "fills in a table by recurrence"; CTC just swaps the values in the table for log probabilities.
5. **One line on the log domain**: multiplying probabilities becomes adding in the log domain, and adding probabilities becomes `logaddexp` ("addition in log space").

```text
        s=0   s=1   s=2   …   (extended label axis)
t=0     α     ·     ·
t=1     ·     ·     ·
…
```

In [ ]:
from itertools import product

toy_labels = [1, 2]  # a b
toy_paths = [
    path
    for path in product(range(3), repeat=3)
    if collapse(path, blank=0) == toy_labels
]
print("Number of legal paths for T=3:", len(toy_paths))
print("The first 5 paths:", toy_paths[:5])
print("The exact sum has to wait for L69's ctc_forward; here in L68 we just look at how the path space grows.")

## Wrap-up

This section implemented `ctc_greedy_decode`, which outputs a `list[int]` (the collapsed token-id sequence), an approximately optimal solution over CTC paths.
The function serves the `aurora.speech` inference pipeline directly — once training with `torch.nn.CTCLoss` is done, greedy decoding is the fastest way to decode in production.
Appendix C of this lesson has already warmed up L69's forward algorithm: look at the path space first, then the log-domain recurrence, and you won't suddenly fall off a cliff in the next section.
The next section (L69) implements the full dynamic program of the CTC forward algorithm, tracking the two kinds of legal transitions — blank-skip and label collapse — to compute the sequence probability.
After L69 we'll also discuss why Whisper chose cross-entropy instead of CTC.

## ✏️ Whiteboard challenge: CTC decoding by hand (target: 10 minutes)

Cover the screen and work it out on paper:

**Question 1**: What target sequence does the path `[a, ∅, a, a, ∅, b]` collapse to?
(remove adjacent repeats: [a,∅,a,∅,b]; remove blanks: [a,a,b])

**Question 2**: Which CTC paths of length 4 correspond to the target sequence `"ab"`? (blank=0, a=1, b=2)
(the a's must come before the b's, with blanks allowed in between; e.g. [a,a,∅,b], [a,∅,b,b], [∅,a,b,b], [a,∅,∅,b], etc.)

**Question 3**: What's the difference between greedy decoding and optimal decoding? When does greedy decoding go wrong?
(greedy = argmax per frame; greedy fails when the best path needs "the second-best token in this frame, then the best one in the next"; beam search does better)

**Question 4**: Why doesn't CTC need frame-level labels (forced alignment)?
(total probability = the sum of the probabilities of all legal paths; a sequence-level label, "this audio goes with this sentence", is enough to train)

**Question 5**: If the logits have shape=(50, 30), what's the longest possible output of greedy decoding?
(at most 50 tokens, if every frame is a different non-blank token from its neighbor; at least 0, if every frame is blank — at most 1 if every frame is the same non-blank token)

When you've finished, run the cell below to check.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: path collapse
path_q1 = [1, 0, 1, 1, 0, 2]  # a=1,blank=0,b=2
result_q1 = collapse(path_q1, blank=0)
assert result_q1 == [1, 1, 2], f"got {result_q1}, expected [1,1,2]"
print(f"Q1 ✅  [a,∅,a,a,∅,b] collapses to {result_q1} = [a,a,b] (a blank between the two a's, so they don't merge)")

# Q2: enumerate length-4 paths (checks that collapse gives the right output)
target_q2 = [1, 2]  # ab
valid_paths = []
for p0 in range(3):
    for p1 in range(3):
        for p2 in range(3):
            for p3 in range(3):
                path = [p0,p1,p2,p3]
                if collapse(path, blank=0) == target_q2:
                    valid_paths.append(path)
assert len(valid_paths) >= 4, f"found {len(valid_paths)} paths, should be ≥4"
print(f"Q2 ✅  target 'ab' has {len(valid_paths)} CTC paths of length 4 (e.g. {valid_paths[:3]}...)")

# Q3: greedy vs optimal (a demo scenario)
try:
    # build logits where greedy is sub-optimal
    logits_q3 = np.zeros((3, 3))
    logits_q3[0, 0] = 1.0   # frame 0: blank is best
    logits_q3[1, 1] = 0.9   # frame 1: token1 is second best
    logits_q3[1, 0] = 1.0   # frame 1: blank is best
    logits_q3[2, 2] = 1.0   # frame 2: token2 is best
    greedy_path = logits_q3.argmax(axis=1)
    greedy_result = collapse(greedy_path.tolist(), blank=0)
    print(f"Q3 ✅  greedy decoding path={greedy_path.tolist()} → {greedy_result}")
    print(f"       beam search may find a better path (here greedy = optimal; in harder cases they differ)")
except Exception as e:
    print(f"Q3 ✅  greedy = argmax per frame; beam search explores several paths, more accurate but slower")

# Q4: sequence-level training (a concept check — uses ctc_greedy_decode)
try:
    logits_q4 = np.array([[0.1, 5.0, 0.1],  # frame 0: token1 largest
                           [0.1, 0.1, 5.0],  # frame 1: token2 largest
                           [5.0, 0.1, 0.1]]) # frame 2: blank largest
    decoded = ctc_greedy_decode(logits_q4, blank=0)
    assert decoded == [1, 2], f"decoded result={decoded}"
    print(f"Q4 ✅  ctc_greedy_decode: logits→{decoded} (needs only the sequence label [1,2], no frame alignment)")
except (NotImplementedError, TypeError):
    print(f"Q4 ✅  CTC needs no frame-level labels: it optimizes the total probability of all legal paths (check again once implemented)")

# Q5: longest/shortest output
try:
    T_q5, V_q5 = 10, 5
    # longest: every frame cycles through different non-blank tokens
    logits_max = np.zeros((T_q5, V_q5))
    for t in range(T_q5): logits_max[t, (t % (V_q5-1)) + 1] = 1.0  # cycling non-blank tokens
    decoded_max = ctc_greedy_decode(logits_max, blank=0)
    # shortest: all blank
    logits_min = np.zeros((T_q5, V_q5)); logits_min[:, 0] = 1.0
    decoded_min = ctc_greedy_decode(logits_min, blank=0)
    print(f"Q5 ✅  for T=10: at most {len(decoded_max)} tokens out, at least {len(decoded_min)} (all blank)")
except (NotImplementedError, TypeError):
    print(f"Q5 ✅  T frames give at most T tokens (no adjacent repeats), at least 0 (all blank) (check again once implemented)")
print("\n🎉 CTC decoding whiteboard challenge passed!")

In [ ]:
# ✏️ Self-review for this lesson
l68_review = {
    "ctc_blank_understood":      None,  # understand what blank does: lets frames >> characters, supports repeated characters? True/False
    "collapse_rules":            None,  # remember the 2 collapse steps: remove adjacent repeats → remove blanks? True/False
    "greedy_decode_impl":        None,  # ctc_greedy_decode() implemented correctly, 3 test cases pass? True/False
    "no_alignment_needed":       None,  # understand why CTC needs no frame-level alignment labels? True/False
    "whiteboard_passed":         None,  # all 5 whiteboard questions worked by hand? True/False
}

unfilled = [k for k, v in l68_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l68_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L68 fully cleared! On to L69: the CTC forward algorithm')

---

→ **Next lesson** · [L69 · The CTC forward algorithm](L69_ctc_forward.ipynb)

> Next lesson covers **the CTC forward algorithm**: summing over all paths, O(T·S) in pure NumPy. Read Appendix C of this lesson first, then go back over L67's DP table-filling logic.